# OmniBird — single-modality event JEPA on **CIFAR10-DVS**

End-to-end training of OmniBird-JEPA on real event-camera data: the
**CIFAR10-DVS** dataset (Li et al. 2017) — event-camera recordings of the
CIFAR-10 images, 10-class classification.

Sensor resolution: **128 × 128**.

This notebook supports two on-disk layouts; set `DATA_LAYOUT` at the top of
the setup cell:

  - `"omnibird"` — per-clip layout (output of either:
       `python -m datasets.download tonic --name CIFAR10DVS --out <dir>`
       or
       `python -m datasets.download convert_cifar10_dvs --raw <aedat> --out <dir>`
     )  Default.

  - `"tonic_native"` — read the cache that `tonic.datasets.CIFAR10DVS`
     creates directly (no conversion needed; loads via Tonic at runtime).


## 1. Setup

In [ ]:
import os, sys, math, time, copy, ssl, glob
sys.path.insert(0, os.path.abspath('..'))
ssl._create_default_https_context = ssl._create_unverified_context

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt

from omnibird import (
    OmniBirdConfig, OmniBirdEncoder, OmniBirdPredictor,
    orderings_from_batch, TargetCenter, ema_update, make_momentum_schedule,
    gather_target_features, jepa_loss, diag_dict, fmt_diag,
    quick_probe, save_atomic, ensure_dir, short_params, count_params,
    build_loaders,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)

# ─── KNOBS (edit these) ────────────────────────────────────────────────
DATA_LAYOUT = "omnibird"                           # "omnibird" or "tonic_native"
DATA_ROOT   = "./data/cifar10_dvs_omnibird"         # path to your CIFAR10-DVS clips
TRAIN_FRAC  = 0.8                                   # split fraction for train/test
# ────────────────────────────────────────────────────────────────────────

cfg = OmniBirdConfig()
cfg.coord_dim       = 3                  # (x, y, t)
cfg.signal_dim      = 1                  # polarity (in {-1, +1})
cfg.n_events_total  = 2048               # events per sample after sub-sampling
cfg.n_ctx           = 1024               # context block size (~50% of available)
cfg.n_tgt_per_block = 64
cfg.n_pred_blocks   = 4
cfg.n_tgt           = cfg.n_pred_blocks * cfg.n_tgt_per_block   # 256
cfg.side            = 64                 # quantization grid for 3-D serialization
cfg.epochs          = 100
cfg.batch_size      = 32
cfg.lr              = 2e-4
cfg.probe_interval  = 10
cfg.probe_epochs    = 3
cfg.probe_patience  = 5
cfg.log_every       = 25
cfg.ckpt_dir        = "./checkpoints_omnibird_cifar10_dvs"
ensure_dir(cfg.ckpt_dir)

print(f"DEVICE = {DEVICE}")
print(f"events/sample = {cfg.n_events_total}  n_ctx = {cfg.n_ctx}  n_tgt = {cfg.n_tgt}")
print(f"D_model = {cfg.d_model}  block_size = {cfg.block_size}  n_layers = {cfg.n_layers_enc}")
print(f"data_layout = {DATA_LAYOUT}  data_root = {DATA_ROOT}")


## 2. CIFAR10-DVS dataset wrapper

CIFAR10-DVS events are recorded on a 128 × 128 DAVIS sensor with timestamps in
microseconds. We normalize to the OmniBird convention:

    coords  ∈ [-1, +1]³   for (x, y, t)
    signal  ∈ {-1, +1}    for polarity

If you used `python -m datasets.download tonic --name CIFAR10DVS --out <dir>`
or `convert_cifar10_dvs`, the data is already in OmniBird per-clip layout
and `omnibird/data.py` knows how to read it via the wrapper below.


In [ ]:
class CIFAR10DVSFromClips:
    # Reads OmniBird per-clip layout (clip_*/events_0.npy + label_0.txt).
    # Each sample -> (events ndarray (N_raw, 4), label int).
    # events_0.npy columns may use sensor pixel ints + microsecond timestamps
    # + polarity in {0, 1}. Normalize to (x, y, t, p) in [-1, 1]^3 x {-1, +1}.
    def __init__(self, root, sensor_hw=(128, 128), n_classes=10):
        self.root = root
        self.h, self.w = sensor_hw
        self.n_classes = n_classes
        self.clips = sorted(glob.glob(os.path.join(root, "clip_*")))
        if not self.clips:
            raise RuntimeError(
                f"No clip_* dirs found in {root}.\n"
                f"Re-run: python -m datasets.download tonic --name CIFAR10DVS --out {root}"
            )

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):
        clip = self.clips[idx]
        ev = np.load(os.path.join(clip, "events_0.npy")).astype(np.float32)
        # Columns expected: (x_int, y_int, t_us, polarity{0 or 1, or -1/+1})
        if ev.shape[0] == 0:
            ev = np.zeros((1, 4), dtype=np.float32)
        x = ev[:, 0] / max(self.w - 1, 1) * 2.0 - 1.0
        y = ev[:, 1] / max(self.h - 1, 1) * 2.0 - 1.0
        t_raw = ev[:, 2]
        t = (t_raw - t_raw.min()) / max(t_raw.max() - t_raw.min(), 1.0) * 2.0 - 1.0
        p = ev[:, 3]
        if p.max() <= 1.0 and p.min() >= 0.0:        # polarity stored as {0,1} → {-1,+1}
            p = p * 2.0 - 1.0
        out = np.stack([x, y, t, p], axis=1).astype(np.float32)
        with open(os.path.join(clip, "label_0.txt")) as f:
            label = int(f.read().strip())
        return out, label


class CIFAR10DVSFromTonic:
    # Reads tonic's native cache for tonic.datasets.CIFAR10DVS.
    # Lazily resolves tonic at import time so the notebook still runs
    # if tonic isn't installed and DATA_LAYOUT='omnibird' is used.
    def __init__(self, save_to="./data/cifar10_dvs_tonic_cache", train=True):
        import tonic
        try:
            self.ds = tonic.datasets.CIFAR10DVS(save_to=save_to, train=train)
        except TypeError:
            self.ds = tonic.datasets.CIFAR10DVS(save_to=save_to)
        self.h, self.w = self.ds.sensor_size[:2]

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        events, label = self.ds[idx]
        # Tonic events are structured arrays with x, y, t, p
        x = events["x"].astype(np.float32) / max(self.w - 1, 1) * 2.0 - 1.0
        y = events["y"].astype(np.float32) / max(self.h - 1, 1) * 2.0 - 1.0
        t_raw = events["t"].astype(np.float32)
        t = (t_raw - t_raw.min()) / max(t_raw.max() - t_raw.min(), 1.0) * 2.0 - 1.0
        p = events["p"].astype(np.float32)
        if p.max() <= 1.0 and p.min() >= 0.0:
            p = p * 2.0 - 1.0
        return np.stack([x, y, t, p], axis=1), int(label)


# Build the base dataset and split into train / test
if DATA_LAYOUT == "omnibird":
    base = CIFAR10DVSFromClips(DATA_ROOT, sensor_hw=(128, 128))
    n = len(base)
    rng = np.random.RandomState(0)
    perm = rng.permutation(n)
    n_train = int(n * TRAIN_FRAC)
    train_idx, test_idx = perm[:n_train], perm[n_train:]

    class _Subset:
        def __init__(self, base, idx): self.base = base; self.idx = idx
        def __len__(self): return len(self.idx)
        def __getitem__(self, i): return self.base[int(self.idx[i])]

    base_train = _Subset(base, train_idx)
    base_test  = _Subset(base, test_idx)
elif DATA_LAYOUT == "tonic_native":
    base_train = CIFAR10DVSFromTonic(save_to=DATA_ROOT, train=True)
    base_test  = CIFAR10DVSFromTonic(save_to=DATA_ROOT, train=False)
else:
    raise ValueError(f"unknown DATA_LAYOUT: {DATA_LAYOUT}")

print(f"loaded CIFAR10-DVS: train={len(base_train)}  test={len(base_test)}")


## 3. JEPA-ready loaders

In [ ]:
train_loader, train_eval_loader, test_loader = build_loaders(base_train, base_test, cfg, num_workers=2)
print(f"train       : {len(train_loader.dataset)} samples, {len(train_loader)} batches")
print(f"train_eval  : {len(train_eval_loader.dataset)} samples (probe-train, test-mode sampling)")
print(f"test        : {len(test_loader.dataset)} samples")

# Peek
batch = next(iter(train_loader))
for k, v in batch.items():
    if hasattr(v, "shape"):
        print(f"  {k:24s} {tuple(v.shape)}  {v.dtype}")


## 4. Visualize a few clips (one per class)

In [ ]:
classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
# Find one example per class by scanning the test set
example_idx = {}
for i in range(min(len(base_test), 500)):
    ev, lab = base_test[i]
    if lab not in example_idx:
        example_idx[lab] = i
        if len(example_idx) == 10:
            break

if example_idx:
    fig = plt.figure(figsize=(20, 8))
    for k, (lab, i) in enumerate(sorted(example_idx.items())):
        ev, _ = base_test[i]
        ax = fig.add_subplot(2, 5, k+1, projection='3d')
        pos = ev[ev[:,3] > 0]; neg = ev[ev[:,3] < 0]
        ax.scatter(pos[:,0], pos[:,1], pos[:,2], s=1, c='red',  alpha=0.4)
        ax.scatter(neg[:,0], neg[:,1], neg[:,2], s=1, c='blue', alpha=0.4)
        ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t')
        ax.set_title(f"{classes[lab]}  ({ev.shape[0]} ev)")
    plt.tight_layout(); plt.show()
else:
    print("(visualization skipped — could not find one example per class)")


## 5. Models

In [ ]:
context_encoder = OmniBirdEncoder(
    d_model=cfg.d_model, n_layers=cfg.n_layers_enc,
    n_heads=cfg.n_heads, dim_head=cfg.dim_head,
    block_size=cfg.block_size, window=cfg.window,
    n_random=cfg.n_random, n_global=cfg.n_global,
    ffn_mult=cfg.ffn_mult,
    signal_dim=cfg.signal_dim, coord_dim=cfg.coord_dim,
    fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
    serial_orders=cfg.serial_orders,
    reinject_pos=cfg.reinject_pos,
).to(DEVICE)
target_encoder = copy.deepcopy(context_encoder).to(DEVICE)
for q in target_encoder.parameters(): q.requires_grad_(False)
predictor = OmniBirdPredictor(
    d_model=cfg.d_model, d_pred=cfg.d_pred,
    n_layers=cfg.n_layers_pred,
    n_heads=cfg.n_heads_pred, dim_head=cfg.dim_head_pred,
    coord_dim=cfg.coord_dim,
    fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
    ffn_mult=cfg.ffn_mult, pos_symmetric=cfg.predictor_pos_symmetric,
).to(DEVICE)
target_center = TargetCenter(cfg.d_model, momentum=0.9).to(DEVICE)
print(f"context_encoder: {short_params(context_encoder)}  predictor: {short_params(predictor)}")


## 6. Optim + schedules + state

In [ ]:
params = list(context_encoder.parameters()) + list(predictor.parameters())
optimizer = AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
total_steps = cfg.epochs * len(train_loader)
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps)
momentum_gen = make_momentum_schedule(cfg.ema_start, cfg.ema_end, total_steps)

PBB_LAST       = os.path.join(cfg.ckpt_dir, "omnibird_last.pt")
PBB_BEST       = os.path.join(cfg.ckpt_dir, "omnibird_best.pt")
PBB_PROBE_BEST = os.path.join(cfg.ckpt_dir, "omnibird_probe_best.pt")

RESUME = True
history = {"loss": [], "diag_steps": [], "diag_log": [], "probe_accs": []}
start_epoch, best_loss, global_step = 1, float("inf"), 0
best_probe_acc, probe_no_improve = -1.0, 0
m = cfg.ema_start

if RESUME and os.path.exists(PBB_LAST):
    s = torch.load(PBB_LAST, map_location=DEVICE, weights_only=False)
    context_encoder.load_state_dict(s["context_encoder"])
    target_encoder.load_state_dict(s["target_encoder"])
    predictor.load_state_dict(s["predictor"])
    target_center.load_state_dict(s["center"])
    optimizer.load_state_dict(s["opt"]); scheduler.load_state_dict(s["sched"])
    history = s.get("history", history)
    global_step = s.get("global_step", 0)
    best_loss = s.get("best_loss", float("inf"))
    best_probe_acc = s.get("best_probe_acc", -1.0)
    probe_no_improve = s.get("probe_no_improve", 0)
    start_epoch = s["epoch"] + 1
    for _ in range(global_step):
        try: m = next(momentum_gen)
        except StopIteration: m = cfg.ema_end
    print(f"resumed @ ep {s['epoch']}, step {global_step}, best_probe_acc={best_probe_acc:.4f}")
else:
    print("starting fresh.")


## 7. Probe wrapper

In [ ]:
def move_orderings(ords, device):
    return {k: {kk: vv.to(device) for kk, vv in v.items()} for k, v in ords.items()}

def probe_now(num_epochs=cfg.probe_epochs, lr=1e-3):
    return quick_probe(
        context_encoder, train_eval_loader, test_loader,
        d_model=cfg.d_model, n_classes=10,
        num_epochs=num_epochs, lr=lr, weight_decay=1e-4, device=DEVICE,
        use_attn_pool=cfg.probe_use_attn_pool,
    )


## 8. Training loop

In [ ]:
epoch = start_epoch - 1
try:
    for epoch in range(start_epoch, cfg.epochs + 1):
        context_encoder.train(); predictor.train()
        epoch_loss, steps = 0.0, 0
        t0 = time.time()

        for batch in train_loader:
            ctx_s  = batch["ctx_signal"].to(DEVICE)
            ctx_c  = batch["ctx_coords"].to(DEVICE)
            pool_s = batch["pool_signal"].to(DEVICE)
            pool_c = batch["pool_coords"].to(DEVICE)
            tgt_c  = batch["tgt_coords"].to(DEVICE)
            tgt_pp = batch["tgt_pool_pos"].to(DEVICE)
            ctx_o  = move_orderings(orderings_from_batch(batch, "ctx"),  DEVICE)
            pool_o = move_orderings(orderings_from_batch(batch, "pool"), DEVICE)

            with torch.no_grad():
                g_tgt = target_encoder(pool_s, pool_c, pool_o)
                h_tgt_raw = gather_target_features(g_tgt, tgt_pp)
                if cfg.use_centering:
                    target_center.update(h_tgt_raw)
                    h_tgt = F.layer_norm(target_center(h_tgt_raw), (h_tgt_raw.size(-1),))
                else:
                    h_tgt = F.layer_norm(h_tgt_raw, (h_tgt_raw.size(-1),))

            g_ctx  = context_encoder(ctx_s, ctx_c, ctx_o)
            h_pred = predictor(g_ctx, tgt_c, ctx_coords=ctx_c if cfg.predictor_pos_symmetric else None)

            loss = jepa_loss(h_pred, h_tgt, loss_type=cfg.loss_type)
            optimizer.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step(); scheduler.step()

            try: m = next(momentum_gen)
            except StopIteration: m = cfg.ema_end
            ema_update(target_encoder, context_encoder, m)

            global_step += 1; epoch_loss += loss.item(); steps += 1
            if global_step % cfg.log_every == 0:
                d = diag_dict(loss, h_pred, h_tgt, g_ctx, target_center)
                print(fmt_diag(d, global_step, epoch, scheduler.get_last_lr()[0], m))
                history["diag_steps"].append(global_step); history["diag_log"].append(d)

        avg = epoch_loss / max(steps, 1)
        history["loss"].append(avg)
        improved = avg < best_loss
        if improved: best_loss = avg

        ckpt_state = {
            "epoch": epoch, "cfg": cfg.__dict__,
            "context_encoder": context_encoder.state_dict(),
            "target_encoder":  target_encoder.state_dict(),
            "predictor":       predictor.state_dict(),
            "center":          target_center.state_dict(),
            "opt": optimizer.state_dict(), "sched": scheduler.state_dict(),
            "global_step": global_step, "best_loss": best_loss,
            "best_probe_acc": best_probe_acc, "probe_no_improve": probe_no_improve,
            "history": history,
        }
        if improved: save_atomic(ckpt_state, PBB_BEST)
        save_atomic(ckpt_state, PBB_LAST)
        print(f"=== ep {epoch:03d}/{cfg.epochs}  avg_loss={avg:.4f}  m={m:.4f}  {time.time()-t0:.1f}s"
              + ("  ★ new best (saved omnibird_best.pt)" if improved else ""))

        if cfg.probe_interval > 0 and epoch % cfg.probe_interval == 0:
            t_probe = time.time()
            print(f"  → Running {cfg.probe_epochs}-epoch linear probe at epoch {epoch}...")
            acc = probe_now()
            history.setdefault("probe_accs", []).append((epoch, acc))
            print(f"  [probe ep {epoch:03d}]  test_acc = {acc:.4f}  ({time.time()-t_probe:.1f}s)")
            if acc > best_probe_acc:
                best_probe_acc = acc; probe_no_improve = 0
                ckpt_state["best_probe_acc"]   = best_probe_acc
                ckpt_state["probe_no_improve"] = probe_no_improve
                ckpt_state["history"]          = history
                save_atomic(ckpt_state, PBB_PROBE_BEST)
                print(f"  ★ new best probe acc → {best_probe_acc:.4f}  (saved omnibird_probe_best.pt)")
            else:
                probe_no_improve += 1
                pp = cfg.probe_patience if cfg.probe_patience > 0 else "∞"
                print(f"  no probe improvement ({probe_no_improve}/{pp})  best so far = {best_probe_acc:.4f}")
            if cfg.probe_patience > 0 and probe_no_improve >= cfg.probe_patience:
                print(f"\nEarly stopping at epoch {epoch}: best probe acc = {best_probe_acc:.4f}")
                break

    print("\nDone.")
except KeyboardInterrupt:
    print(f"\nInterrupted at epoch {epoch}.  Checkpoints in {cfg.ckpt_dir}.")


## 9. Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
if history["loss"]:
    axes[0].plot(range(1, len(history["loss"])+1), history["loss"], lw=1.5)
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("avg loss"); axes[0].set_title("JEPA loss"); axes[0].grid(alpha=0.3)
if history["diag_log"]:
    steps = history["diag_steps"]
    cos_mean = [d["cos_mean"] for d in history["diag_log"]]
    axes[1].plot(steps, cos_mean, color='C2'); axes[1].set_ylim(-0.1, 1.05)
    axes[1].set_xlabel("step"); axes[1].set_title("cos(h_pred, h_tgt)"); axes[1].grid(alpha=0.3)
if history.get("probe_accs"):
    eps, accs = zip(*history["probe_accs"])
    axes[2].plot(eps, [a*100 for a in accs], 'o-', color='C3', markersize=6)
    axes[2].set_ylim(0, 100)
    axes[2].set_xlabel("epoch"); axes[2].set_ylabel("probe acc (%)")
    axes[2].set_title(f"probe acc (best = {max(accs)*100:.2f}%)"); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 10. Final probe (longer, for a stable number)

In [ ]:
LOAD_PROBE_BEST = True
PROBE_EPOCHS_LONG = 10
PROBE_LR = 1e-3

if LOAD_PROBE_BEST and os.path.exists(PBB_PROBE_BEST):
    s = torch.load(PBB_PROBE_BEST, map_location=DEVICE, weights_only=False)
    context_encoder.load_state_dict(s["context_encoder"])
    target_encoder.load_state_dict(s["target_encoder"])
    print(f"loaded encoders from {PBB_PROBE_BEST}  (ep {s.get('epoch','?')}, best probe={s.get('best_probe_acc',float('nan')):.4f})")
elif LOAD_PROBE_BEST:
    print(f"{PBB_PROBE_BEST} not present — probing the encoder in memory")

print(f"running {PROBE_EPOCHS_LONG}-epoch probe ...")
t0 = time.time()
acc = probe_now(num_epochs=PROBE_EPOCHS_LONG, lr=PROBE_LR)
print(f"\nfinal probe acc(CIFAR10-DVS) = {acc*100:.2f}%   ({time.time()-t0:.1f}s)")

if history.get("probe_accs"):
    eps, accs = zip(*history["probe_accs"])
    print(f"\nduring-training trace ({len(accs)} probes):")
    for e, a in history["probe_accs"]:
        marker = "  <-- best" if a == max(accs) else ""
        print(f"  ep {e:3d}:  {a*100:5.2f}%{marker}")
    print(f"\nfinal vs best in-training: {acc*100:.2f}% vs {max(accs)*100:.2f}% "
          f"({(acc - max(accs))*100:+.2f} pp)")
